# Discriminating experiment: order sweep vs CV-1SE in the noise band 
Both selectors run on IDENTICAL datasets (each selector uses its own
resampling scheme; no claim of shared train/test partitions is made):
o3 (pure monomial) at pair-0.9, n = 20,000, sigma in {1.0, 1.5, 2.0, 2.5, 3.0},
R = 20 replicates. Rows carry the data seed and a sha256 of (X, h) so the
pairing is demonstrated in the artifact, not assumed.

The selection rule is FIXED-SEQUENCE (hierarchical) testing on the ordered
remaining-gap hypotheses: stop at the first accepted H_k. Its overselection
guarantee P(khat > true order) <= alpha is exact. The verification cell
audits report coherence (rows with any p_j <= alpha above khat).

PRE-REGISTERED prediction (recorded as a prediction, checked in OBS): the
population MSE gap between C_2 and C_3 is F(0.9) x Var(h) = 0.0199,
independent of sigma; the CV-1SE cutoff grows with the noise level. A
HEURISTIC APPROXIMATION of the crossover -- treating fold MSE variability
as pure Gaussian-noise variability and ignoring fold overlap, signal
residual terms, coefficient-estimation noise, and the randomness of the
best order -- gives sigma* ~ 1.4, but preliminary runs show the approximation sits low
(the ignored terms shrink the realized margin). Prediction, band form:
CV-1SE's rate of selecting 3 decreases to zero as sigma grows across the
grid while the order sweep's stays high; the empirical crossover is
bracketed by the grid and reported wherever it falls. A failed prediction
is reported as such.

Acceptance checks (PASS/FAIL, provable): (1) completeness and demonstrated
pairing on (sigma, rep, data_seed, data_hash); (2) bias: the measured
remaining gap at k = 2 matches the OPTIMISM-ADJUSTED closed form within
max(4 SE, 1e-4) at every sigma (SE with ddof=1 over replicates; the
adjustment is the validated (p_K - p_k)(1 - R^2_K)/n_tr allowance).
Estimator precision (seed sd) is reported separately as an observation and
does not enlarge the acceptance region.

Reruns refuse to overwrite an existing artifact unless OVERWRITE = True.
Outputs to `MyDrive/ORDER_SWEEP/results/discriminator_power/`.


In [ ]:
# Cell 1 -- Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/ORDER_SWEEP'
OUT = os.path.join(BASE, 'results', 'discriminator_power')
os.makedirs(OUT, exist_ok=True)
print('output folder:', OUT)


In [ ]:
# Cell 2 -- Self-contained machinery (fixed-sequence selector + CV-1SE), source-hashed
import numpy as np, json, csv, time, hashlib, zlib, sys, platform, inspect
import scipy
from itertools import product as iproduct
from scipy import stats

def monomial_exps(d, D, max_active):
    out = []
    for combo in iproduct(range(D + 1), repeat=d):
        if sum(combo) <= D and sum(1 for c in combo if c > 0) <= max_active:
            out.append(combo)
    return out

def poly_design(X, D, max_active):
    d = X.shape[1]
    exps = monomial_exps(d, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    return np.column_stack(cols), exps.index(tuple([0] * d))

def build_designs(X, K, D=4):
    """Design matrices depend only on (X, D, k): build once per dataset."""
    return {k: poly_design(X, D, k) for k in range(1, K + 1)}

def holdout_r2_nested(X, h, orders, split_seed, designs, train_frac=0.75):
    n = X.shape[0]
    idx = np.random.default_rng(split_seed).permutation(n)
    tr, te = idx[: int(train_frac * n)], idx[int(train_frac * n):]
    out = {}
    for k in orders:
        Phi0, ci = designs[k]
        mu = Phi0[tr].mean(0); sd = Phi0[tr].std(0); sd[sd == 0] = 1.0
        Phi = (Phi0 - mu) / sd
        Phi[:, ci] = 1.0
        hm = h[tr].mean()
        beta, *_ = np.linalg.lstsq(Phi[tr], h[tr] - hm, rcond=None)
        resid = (h[te] - hm) - Phi[te] @ beta
        denom = np.sum((h[te] - h[te].mean()) ** 2)
        out[k] = 1.0 - float((resid @ resid) / denom)
    return out

def select_order(X, h, K=3, S=10, alpha=0.05, D=4, train_frac=0.75, designs=None):
    """Order-sweep selection by FIXED-SEQUENCE (hierarchical) testing on the
    ordered remaining-gap hypotheses H_k ("no component above order k"),
    tested via the paired holdout gap C_k -> C_K across S shared splits
    with the Nadeau-Bengio corrected resampled t-statistic; stop at the
    first accepted H_k. P(khat > true order) <= alpha exactly. Certificate:
    one-sided upper bound on the remaining gap at khat plus the
    fitting-optimism allowance (p_K - p_k)(1 - R^2_K)/n_tr."""
    n = X.shape[0]
    n_tr = int(train_frac * n); n_te = n - n_tr
    if designs is None:
        designs = build_designs(X, K, D)
    r2 = {s: holdout_r2_nested(X, h, range(1, K + 1), s, designs, train_frac)
          for s in range(S)}
    corr = 1.0 / S + n_te / n_tr
    p_feat = {k: designs[k][0].shape[1] for k in range(1, K + 1)}
    one_minus_r2K = float(np.mean([1.0 - r2[s][K] for s in range(S)]))
    tcrit = stats.t.ppf(1 - alpha, df=S - 1)
    stat = {}
    for k in range(1, K):
        g = np.array([r2[s][K] - r2[s][k] for s in range(S)])
        m = g.mean()
        v = g.var(ddof=1) * corr
        opt = (p_feat[K] - p_feat[k]) * one_minus_r2K / n_tr
        if v > 0:
            t = m / np.sqrt(v)
            p = 1.0 - stats.t.cdf(t, df=S - 1)
            ub = m + tcrit * np.sqrt(v) + opt
        else:
            # degenerate zero-variance branch: evidence direction follows the sign
            t = np.inf if m > 0 else (-np.inf if m < 0 else 0.0)
            p = 0.0 if m > 0 else 1.0
            ub = m + opt
        stat[k] = {"mean": float(m), "p": float(p), "ub": float(ub),
                   "pi": float((g > 0).mean())}
    khat, ub_cert = K, None
    for k in range(1, K):
        if stat[k]["p"] > alpha:
            khat, ub_cert = k, stat[k]["ub"]
            break
    return khat, stat, ub_cert

def cv1se_khat_diag(X, h, K=3, D=4, folds=5, seed=0, designs=None):
    """CV-1SE order selection on the same classes. Returns khat, per-order
    CV means, the 1SE margin (SE of the best order's fold MSEs), and the
    full cutoff (best mean + margin)."""
    n = X.shape[0]
    if designs is None:
        designs = build_designs(X, K, D)
    idx = np.random.default_rng(seed).permutation(n)
    fold_id = np.empty(n, dtype=int)
    fold_id[idx] = np.arange(n) % folds
    errs = {k: [] for k in range(1, K + 1)}
    for k in range(1, K + 1):
        Phi0, ci = designs[k]
        for f in range(folds):
            tr, te = fold_id != f, fold_id == f
            mu = Phi0[tr].mean(0); sd = Phi0[tr].std(0); sd[sd == 0] = 1.0
            P = (Phi0 - mu) / sd; P[:, ci] = 1.0
            hm = h[tr].mean()
            beta, *_ = np.linalg.lstsq(P[tr], h[tr] - hm, rcond=None)
            resid = (h[te] - hm) - P[te] @ beta
            errs[k].append(float(np.mean(resid ** 2)))
    means = {k: float(np.mean(v)) for k, v in errs.items()}
    ses = {k: float(np.std(v, ddof=1) / np.sqrt(folds)) for k, v in errs.items()}
    kbest = min(means, key=means.get)
    margin = ses[kbest]
    cutoff = means[kbest] + margin
    khat = min(k for k in range(1, K + 1) if means[k] <= cutoff)
    return khat, means, margin, cutoff

def make_pair09(n, rng):
    x1, x2, z = rng.standard_normal((3, n))
    return np.column_stack([x1, x2, 0.9 * x1 + np.sqrt(1 - 0.81) * z])

N = 20_000
R_REPS = 20
SIGMAS = [1.0, 1.5, 2.0, 2.5, 3.0]
SEED_SCHEME = "crc32-full-v3"
OVERWRITE = False
_config = {"N": N, "R_REPS": R_REPS, "SIGMAS": SIGMAS, "dgp": "o3",
           "dependence": "pair0.9", "S_SPLITS": 10, "ALPHA": 0.05,
           "K_MAX": 3, "D": 4, "train_frac": 0.75, "cv_folds": 5,
           "SEED_SCHEME": SEED_SCHEME}
def _fn_repr(f):
    """Function source when available (notebooks are source-backed); else
    bytecode plus constants and names, which unlike co_code alone changes
    when a numeric or string literal changes."""
    try:
        return inspect.getsource(f), True
    except OSError:
        c = f.__code__
        return repr((c.co_code, c.co_consts, c.co_names, c.co_varnames)), False

_parts = [_fn_repr(f) for f in
          [monomial_exps, poly_design, build_designs, holdout_r2_nested,
           select_order, cv1se_khat_diag, make_pair09]]
PROV_SCHEME = "source-v2" if all(ok for _, ok in _parts) else "code+consts-v2"
CODE_SHA = hashlib.sha256("\n\n".join(s for s, _ in _parts).encode()
    + json.dumps(_config, sort_keys=True).encode()).hexdigest()
print(f"provenance sha256 ({PROV_SCHEME}):", CODE_SHA)


In [ ]:
# Cell 3 -- Paired run (data identity recorded per row; no silent overwrite)
EXP = "discriminator_power"
PS = os.path.join(OUT, "per_seed.csv")
if os.path.exists(PS) and not OVERWRITE:
    raise RuntimeError(f"{PS} exists; set OVERWRITE = True in Cell 2 to replace it")

def data_seed(sigma, rep):
    return (90_000 + zlib.crc32(f"disc|{sigma:.17g}".encode()) + rep * 977) % 2**32

t0 = time.time()
rows = []
for sigma in SIGMAS:
    for rep in range(R_REPS):
        ds = data_seed(sigma, rep)
        rng = np.random.default_rng(ds)
        X = make_pair09(N, rng)
        h = X[:, 0] * X[:, 1] * X[:, 2] + sigma * rng.standard_normal(N)
        dh = hashlib.sha256(X.astype(np.float64).tobytes()
                            + h.astype(np.float64).tobytes()).hexdigest()
        designs = build_designs(X, K=3, D=4)
        k_os, stat, ub = select_order(X, h, designs=designs)
        k_cv, cv_means, cv_margin, cv_cutoff = cv1se_khat_diag(X, h, seed=rep, designs=designs)
        rows.append({"experiment": EXP, "sigma": sigma, "rep": rep,
                     "data_seed": ds, "data_hash": dh,
                     "khat_ordersweep": k_os, "khat_cv1se": k_cv,
                     "rem1_p": stat[1]["p"], "rem2_p": stat[2]["p"],
                     "rem2_mean": stat[2]["mean"],
                     "ub_cert": "" if ub is None else ub,
                     "cv_mse_gap23": cv_means[2] - cv_means[3],
                     "cv_1se_margin": cv_margin, "cv_1se_cutoff": cv_cutoff})
    print(f"sigma={sigma:3.1f} done ({time.time()-t0:5.0f}s)", flush=True)

with open(PS, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump({"experiment": EXP, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "config": _config, "code_sha256": CODE_SHA,
               "provenance_scheme": PROV_SCHEME,
               "provenance_scope": "full function source + config; seed scheme in config",
               "numpy": np.__version__, "scipy": scipy.__version__,
               "python": sys.version, "platform": platform.platform(),
               "blas": str(np.__config__.get_info("blas_opt_info") if hasattr(np.__config__, "get_info") else "see numpy.show_config")},
              f, indent=2)
print("wrote per_seed.csv, metadata.json")


In [ ]:
# Cell 4 -- Verification: demonstrated pairing, bias vs adjusted closed form,
# precision, coherence audit, crossover observations
import csv as _csv
rows = list(_csv.DictReader(open(os.path.join(OUT, "per_seed.csv"))))
assert all(r["experiment"] == "discriminator_power" for r in rows), "stamp mismatch"

F09 = (1 - 0.81) ** 2 / ((1 + 0.81) * (1 + 1.62))
VH = 1 + 2 * 0.81
P3_MINUS_P2 = 35 - 31        # feature counts of C_3 vs C_2 (d=3, D=4)
N_TR = int(0.75 * 20_000)
checks, story = [], []

keys = set((r["sigma"], r["rep"]) for r in rows)
hashes_ok = all(len(r["data_hash"]) == 64 and r["data_seed"] for r in rows)
checks.append((f"completeness and demonstrated pairing: {len(rows)} rows (expect 100), "
               f"{len(keys)} unique (sigma, rep) keys, data_seed and data_hash recorded on every row",
               len(rows) == 100 and len(keys) == 100 and hashes_ok))

gap_ok = True
print(f"{'sigma':>6s}{'adj truth':>11s}{'measured rem2':>20s}{'os: k3':>8s}{'cv: k3':>8s}"
      f"{'cv MSE gap':>12s}{'cv cutoff-best':>15s}")
for sigma in [1.0, 1.5, 2.0, 2.5, 3.0]:
    cs = [r for r in rows if float(r["sigma"]) == sigma]
    tg = F09 * VH / (VH + sigma ** 2)
    one_minus_r2 = sigma ** 2 / (VH + sigma ** 2)
    tg_adj = tg - P3_MINUS_P2 * one_minus_r2 / N_TR      # optimism-adjusted reference
    v = [float(r["rem2_mean"]) for r in cs]
    m = float(np.mean(v)); sd = float(np.std(v, ddof=1)); se = sd / np.sqrt(len(v))
    if abs(m - tg_adj) > max(4 * se, 1e-4):
        gap_ok = False
    os3 = sum(int(r["khat_ordersweep"]) == 3 for r in cs)
    cv3 = sum(int(r["khat_cv1se"]) == 3 for r in cs)
    g23 = float(np.mean([float(r["cv_mse_gap23"]) for r in cs]))
    mrg = float(np.mean([float(r["cv_1se_margin"]) for r in cs]))
    print(f"{sigma:6.1f}{tg_adj:11.5f}{m:13.5f}±{sd:.5f}{os3:5d}/{len(cs)}{cv3:5d}/{len(cs)}"
          f"{g23:12.5f}{mrg:15.5f}")
    story.append(f"OBS   sigma={sigma}: estimator precision (seed sd of rem2) = {sd:.5f}")
checks.append(("bias: measured rem2 matches the optimism-adjusted closed form within "
               "max(4 SE, 1e-4) at every sigma", gap_ok))

incoh = [r for r in rows if int(r["khat_ordersweep"]) == 1 and float(r["rem2_p"]) <= 0.05]
checks.append((f"coherence audit: {len(incoh)} rows select k=1 while p_2 <= alpha "
               "(fixed-sequence incoherent-report pattern; 0 expected here since rem_1 ~ 1)",
               len(incoh) == 0))
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)

story.append("OBS   heuristic crossover approximation sigma* ~ 1.4 (ignores fold overlap, "
             "signal residuals, coefficient noise, and best-order randomness)")
for sigma in [1.0, 1.5, 2.0, 2.5, 3.0]:
    cs = [r for r in rows if float(r["sigma"]) == sigma]
    story.append(f"OBS   sigma={sigma}: ordersweep k=3 in "
                 f"{sum(int(r['khat_ordersweep'])==3 for r in cs)}/{len(cs)}, "
                 f"CV-1SE k=3 in {sum(int(r['khat_cv1se'])==3 for r in cs)}/{len(cs)} "
                 f"(band prediction: CV-1SE k3-rate decreasing in sigma, ordersweep persistent)")
for line in story:
    if line.startswith("OBS"):
        print(line)
with open(os.path.join(OUT, "check.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote check.txt")
